In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==============================================================================
# PASSO 1 — Diagnóstico (Aula 09)
# ==============================================================================
print("--- PASSO 1: DIAGNÓSTICO ---")
df_bruto = pd.read_csv('MetObjects.csv', low_memory=False)

print(f"Total de linhas brutas: {len(df_bruto)}")
print(f"Total de duplicatas encontradas: {df_bruto.duplicated().sum()}")

print("\nValores nulos nas colunas principais:")
cols_interesse = ['Department', 'Is Highlight', 'Is Public Domain', 'AccessionYear', 'Object End Date']
print(df_bruto[cols_interesse].isnull().sum())

print("\nDistribuição de categorias em 'Department' (Top 5):")
print(df_bruto['Department'].value_counts().head(5))

print("\nResumo estatístico numérico (Object End Date e AccessionYear):")
print(df_bruto[['Object End Date', 'AccessionYear']].describe())


# ==============================================================================
# PASSO 2 — Limpeza e Filtragem de Outliers (Aula 09)
# ==============================================================================
print("\n--- PASSO 2: LIMPEZA E TRATAMENTO DE OUTLIERS ---")
# Removendo duplicatas
df_limpo = df_bruto.drop_duplicates(keep='first').reset_index(drop=True)

# Padronizando a string da coluna categórica 'Department'
df_limpo['Department'] = df_limpo['Department'].astype(str).str.strip().str.title()

# Convertendo colunas alvo e explicativas para numérico
df_limpo['Object End Date'] = pd.to_numeric(df_limpo['Object End Date'], errors='coerce')
df_limpo['AccessionYear'] = pd.to_numeric(df_limpo['AccessionYear'], errors='coerce')

# Removendo anomalias extremas / outliers de anos (ex: datas irreais ou futuras)
df_limpo.loc[df_limpo['Object End Date'] < -3000, 'Object End Date'] = np.nan
df_limpo.loc[df_limpo['Object End Date'] > 2026, 'Object End Date'] = np.nan

# Removendo linhas onde o alvo (Object End Date) é nulo para o treino
df_modelagem_base = df_limpo.dropna(subset=['Object End Date']).copy()
print(f"Registros válidos para modelagem após limpeza: {len(df_modelagem_base)}")


# ==============================================================================
# PASSO 3 — Preparação Preditiva Livre de Data Leakage (Aula 10)
# ==============================================================================
print("\n--- PASSO 3: PREPARAÇÃO PREDITIVA ---")

# Selecionando colunas relevantes para o modelo
features_cat = ['Department']
features_num = ['Is Highlight', 'Is Public Domain', 'AccessionYear']
target = 'Object End Date'

subset_reg = df_modelagem_base[features_cat + features_num + [target]].copy()

# One-Hot Encoding na coluna categórica 'Department'
df_encoded = pd.get_dummies(subset_reg, columns=features_cat, drop_first=True, dtype=int)

# Definindo X (Matriz Explicativa) e y (Vetor Alvo)
X = df_encoded.drop(columns=[target])
y = df_encoded[target]

# Divisão Treino e Teste (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Tamanho do Treino: {len(X_train)} | Tamanho do Teste: {len(X_test)}")

# Calculando a mediana APENAS no X_train para evitar Data Leakage
medianas_treino = X_train.median()
print("\nMedianas calculadas exclusivamente no conjunto de Treino:")
print(medianas_treino[['AccessionYear']])

# Aplicando a imputação de NaNs no Treino e no Teste usando as medianas do Treino
X_train_prep = X_train.fillna(medianas_treino)
X_test_prep = X_test.fillna(medianas_treino)


# ==============================================================================
# PASSO 4 — Regressão Linear Simples vs. Múltipla (Aula 10)
# ==============================================================================
print("\n--- PASSO 4: REGRESSÃO SIMPLES VS. MÚLTIPLA ---")

# --- 4.1 Regressão Simples (Usando apenas AccessionYear) ---
X_train_simples = X_train_prep[['AccessionYear']]
X_test_simples = X_test_prep[['AccessionYear']]

reg_simples = LinearRegression()
reg_simples.fit(X_train_simples, y_train)
y_pred_simples = reg_simples.predict(X_test_simples)

mae_s = mean_absolute_error(y_test, y_pred_simples)
rmse_s = np.sqrt(mean_squared_error(y_test, y_pred_simples))
r2_s = r2_score(y_test, y_pred_simples)

print(">> Regressão Simples (AccessionYear):")
print(f"   - MAE:  {mae_s:.2f} anos")
print(f"   - RMSE: {rmse_s:.2f}")
print(f"   - R²:   {r2_s:.4f} ({r2_s*100:.1f}%)")

# --- 4.2 Regressão Múltipla (Usando todas as variáveis) ---
reg_multipla = LinearRegression()
reg_multipla.fit(X_train_prep, y_train)
y_pred_multipla = reg_multipla.predict(X_test_prep)

mae_m = mean_absolute_error(y_test, y_pred_multipla)
rmse_m = np.sqrt(mean_squared_error(y_test, y_pred_multipla))
r2_m = r2_score(y_test, y_pred_multipla)

print("\n>> Regressão Múltipla (Todas as variáveis):")
print(f"   - MAE:  {mae_m:.2f} anos")
print(f"   - RMSE: {rmse_m:.2f}")
print(f"   - R²:   {r2_m:.4f} ({r2_m*100:.1f}%)")


# ==============================================================================
# PASSO 5 — Interpretação de Negócio (Aula 10)
# ==============================================================================
print("\n--- PASSO 5: INTERPRETAÇÃO DE NEGÓCIO DOS COEFICIENTES ---")
df_pesos = pd.DataFrame({
    'Atributo (Feature)': X_train_prep.columns,
    'Peso (Beta_i)': reg_multipla.coef_.round(3)
})

print(df_pesos.sort_values(by='Peso (Beta_i)', ascending=False).head(10))

--- PASSO 1: DIAGNÓSTICO ---
Total de linhas brutas: 502949
Total de duplicatas encontradas: 0

Valores nulos nas colunas principais:
Department             0
Is Highlight           0
Is Public Domain       0
AccessionYear       4118
Object End Date        0
dtype: int64

Distribuição de categorias em 'Department' (Top 5):
Department
Drawings and Prints                       185147
European Sculpture and Decorative Arts     43977
Photographs                                39581
Asian Art                                  37325
Greek and Roman Art                        33629
Name: count, dtype: int64

Resumo estatístico numérico (Object End Date e AccessionYear):
       Object End Date
count    502949.000000
mean       1414.090888
std        1495.835243
min     -240000.000000
25%        1599.000000
50%        1844.000000
75%        1909.000000
max        9999.000000

--- PASSO 2: LIMPEZA E TRATAMENTO DE OUTLIERS ---
Registros válidos para modelagem após limpeza: 501976

--- PASSO 3: PRE